QPE without QECC, purely on the physical level

In [2]:
from qiskit import __version__
print(__version__)

2.1.1


In [29]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector, state_fidelity, Pauli, DensityMatrix, partial_trace
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from qiskit_aer.library import SaveDensityMatrix
from qiskit import transpile 
import numpy as np
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit.circuit.library import RZGate, RYGate, UnitaryGate
import matplotlib.pyplot as plt
import random
import math
import time
import pickle
from typing import List

In [16]:
theta = np.arctan(np.sqrt((np.sqrt(5) - 1) / 2))

# Numpy Gates

In [14]:
def rz_gate(theta: float):
    return np.array([[np.exp(-1j * theta/2), 0], [0, np.exp(1j * theta/2)]])

In [17]:
pauli_x = np.array([[0, 1], [1, 0]])
pauli_y = np.array([[0, -1j], [1j, 0]])
pauli_z = np.array([[1, 0], [0, -1]])
hadamard = 1/np.sqrt(2) * np.array([[1, 1], [1, -1]])
s_gate = rz_gate(np.pi/2)
sdg_gate = rz_gate(-np.pi/2)
T = np.array([[1,0], [0,np.exp(1j*np.pi/4)]])
Tdg = np.array([[1,0], [0,np.exp(-1j*np.pi/4)]])
ry_theta = np.array([[np.cos(theta/2), -np.sin(theta/2)], [np.sin(theta/2), np.cos(theta/2)]])

# Infidelity to Error Rates

In [4]:
def inf_to_error(infidelity_list, num_qubits):
    error_rates_list = []
    dimension = 2**num_qubits
    for infidelity in infidelity_list:
        error_rates_list.append( infidelity * (dimension / (dimension - 1)) )
    
    return error_rates_list

In [5]:
one_qubit_gate_infidelity = [2.8e-5 * i for i in range(1,6)]
two_qubit_gate_infidelity = [8.3e-4 * i for i in range(1,6)]
idle_gate_infidelity = [1.2e-4 * i for i in range(1,6)]
spam0 = [6.7e-4 * i for i in range(1,6)] # P(1|0), measured 1 given that 0 was prepared
spam1 = [1.2e-3 * i for i in range(1,6)] # P(0|1)

In [6]:
one_qubit_gate_error_prob = inf_to_error(one_qubit_gate_infidelity, num_qubits=1)
two_qubit_gate_error_prob = inf_to_error(two_qubit_gate_infidelity, num_qubits=2)
idle_gate_error_prob = inf_to_error(idle_gate_infidelity, num_qubits=1)

# Importing Cluster Results

In [67]:
with open("approx_results.pkl", "rb") as f:
    data = pickle.load(f)

approx_gates = data["approx_gates"]
saved_beta_list = data["beta_list"]

In [36]:
name_to_gate = {
    'X': pauli_x,
    'Y': pauli_y, 
    'Z': pauli_z,
    'H': hadamard,
    'S': s_gate,
    'Sdg': sdg_gate,
    'Rz(1*π/4)': T,
    'Rz(-1*π/4)': Tdg,
    'EMS': ry_theta
}

In [68]:
print(len(approx_gates))
print(len(saved_beta_list))

1006
1000


# Functions

In [ ]:
def state_prep(qc: QuantumCircuit, alpha0: float, alpha1: float):
    qc.h(0)
    qc.x(1)
    qc.ry(alpha0, 1)
    qc.rz(alpha1, 1)
    
    
def approx_state_prep(qc: QuantumCircuit, alpha0: float, alpha1: float):
    temp0 = np.eye(2)
    temp1 = np.eye(2)
    for i in approx_gates[0]: 
        temp0 = name_to_gate[i] @ temp0
    for i in approx_gates[1]:
        temp1 = name_to_gate[i] @ temp1
    gate0 = UnitaryGate(temp0, label="gate0")
    gate1 = UnitaryGate(temp1, label="gate1")
    qc.h(0)
    qc.x(1)
    qc.append(gate0, [1])
    qc.append(gate1, [1])

In [60]:
def control_u(qc: QuantumCircuit, h1: float, h2: float, t: float):
    qc.rz(h1*t, 1)
    qc.cx(0,1)
    qc.rz(-h1*t, 1)
    qc.cx(0,1)
    qc.h(1)
    qc.rz(h2*t, 1)
    qc.cx(0,1)
    qc.rz(-h2*t, 1)
    qc.cx(0,1)
    qc.h(1)
    
def approx_control_u(qc: QuantumCircuit, h1: float, h2: float, t: float):
    temp2 = np.eye(2)
    temp3 = np.eye(2)
    temp4 = np.eye(2)
    temp5 = np.eye(2)
    for i in approx_gates[2]:
        temp2 = name_to_gate[i] @ temp2
    for i in approx_gates[3]:
        temp3 = name_to_gate[i] @ temp3
    for i in approx_gates[4]:
        temp4 = name_to_gate[i] @ temp4
    for i in approx_gates[5]:
        temp5 = name_to_gate[i] @ temp5
    gate2 = UnitaryGate(temp2, label="gate2")
    gate3 = UnitaryGate(temp3, label="gate3")
    gate4 = UnitaryGate(temp4, label="gate4")
    gate5 = UnitaryGate(temp5, label="gate5")
    
    qc.append(gate2, [1])
    qc.cx(0,1)
    qc.append(gate3, [1])
    qc.cx(0,1)
    qc.h(1)
    qc.append(gate4, [1])
    qc.cx(0,1)
    qc.append(gate5, [1])
    qc.cx(0,1)
    qc.h(1)

In [61]:
def qpe(qc: QuantumCircuit, alpha0: float, alpha1: float, h1: float, h2: float, t: float, beta: float, meas_bit: ClassicalRegister, k: int):
    state_prep(qc, alpha0, alpha1)
    for _ in range(k):
        control_u(qc, h1, h2, t)
    
    qc.rz(beta, 0)
    qc.h(0)
    qc.measure(0, meas_bit)
    
def approx_qpe(qc: QuantumCircuit, alpha0: float, alpha1: float, h1: float, h2: float, t: float, beta_list_idx: int, meas_bit: ClassicalRegister, k: int):
    temp6 = np.eye(2)
    for i in approx_gates[beta_list_idx+6]:
        temp6 = name_to_gate[i] @ temp6
    gate6 = UnitaryGate(temp6, label="gate6")
    
    approx_state_prep(qc, alpha0, alpha1)
    for _ in range(k):
        approx_control_u(qc, h1, h2, t)
    
    qc.append(gate6, [0])
    qc.h(0)
    qc.measure(0, meas_bit)
        

In [55]:
rng = random.Random(time.time_ns())

h1, h2, h3 = (0.79605, -0.18092, -0.32096)
alpha0, alpha1 = (-0.274220, -0.785398)
t = np.pi / (8*h1)
qc = QuantumCircuit(2)

In [56]:
def distance(U, V):
    # U and V are 2x2 complex matrices
    if np.allclose(U, V):
        return 0.0
    else:
        overlap = 0.5 * np.trace(U.conj().T @ V)
        return np.sqrt((1 - abs(overlap)) + 0j).real

In [79]:
# Ideal version
kmax = 50
num_repeat_circuit = 1_000
meas_list = []
beta_list = []
k_list = []
for _ in range(num_repeat_circuit):
    noise_model = NoiseModel()

    meas_circuit = qc.copy()
    
    beta = rng.uniform(0, math.pi / 2)
    k = rng.randint(1, kmax)
    beta_list.append(beta)
    k_list.append(k)
    
    meas_bit = ClassicalRegister(1)
    meas_circuit.add_register(meas_bit)
    qpe(meas_circuit, alpha0, alpha1, h1, h2, t, beta, meas_bit, k)

    backend = AerSimulator(noise_model=noise_model, method='statevector')
    job = backend.run(meas_circuit, shots=1)
    result = job.result()
    counts = result.get_counts()

    meas_list.append(int(list(counts.keys())[0]))

In [ ]:
# Approximated version
kmax = 50
num_repeat_circuit = 1_000
meas_list = []
k_list = []
for i in range(num_repeat_circuit):
    noise_model = NoiseModel()

    meas_circuit = qc.copy()
    
    k = rng.randint(1, kmax)
    k_list.append(k)
    
    meas_bit = ClassicalRegister(1)
    meas_circuit.add_register(meas_bit)
    approx_qpe(meas_circuit, alpha0, alpha1, h1, h2, t, i, meas_bit, k)

    backend = AerSimulator(noise_model=noise_model, method='statevector')
    job = backend.run(meas_circuit, shots=1)
    result = job.result()
    counts = result.get_counts()

    meas_list.append(int(list(counts.keys())[0]))

In [9]:
def estimate_phase(meas_list, beta_list, k_list, phi_list):
    logL_list = []

    for phi in phi_list:
        logL = 0.0
        for meas, beta, k in zip(meas_list, beta_list, k_list):
            p = (1 + np.cos(k*phi + beta - meas*np.pi)) / 2
            p = np.clip(p, 1e-15, 1.0)
            logL += np.log(p)

        logL_list.append(logL)

    idx = int(np.argmax(logL_list))
    return phi_list[idx], logL_list[idx], logL_list

Estimating the ideal case

In [80]:
phi_list = np.linspace(0, 2*np.pi, 100000, endpoint=False)
phi_hat, logL_max, logL_list = estimate_phase(meas_list, beta_list, k_list, phi_list)
ground_state_energy = h3 - phi_hat/t

In [12]:
print(ground_state_energy)

-1.1348415200000002


In [31]:
H = h1*np.array([[1,0], [0,-1]]) + h2*np.array([[0,1], [1,0]]) + h3*np.eye(2, dtype='complex')

eigenvalues, eigenvectors = np.linalg.eigh(H)
actual_ground_state_energy = eigenvalues[0]
print(actual_ground_state_energy)

-1.137310199914228


In [32]:
print(abs(actual_ground_state_energy-ground_state_energy))

0.0024686799142279447


Estimating the approximation case

In [73]:
phi_list = np.linspace(0, 2*np.pi, 100000, endpoint=False)
phi_hat, logL_max, logL_list = estimate_phase(meas_list, saved_beta_list, k_list, phi_list)
ground_state_energy = h3 - phi_hat/t

In [75]:
print(ground_state_energy)

-1.1350962559999997


In [76]:
H = h1*np.array([[1,0], [0,-1]]) + h2*np.array([[0,1], [1,0]]) + h3*np.eye(2, dtype='complex')

eigenvalues, eigenvectors = np.linalg.eigh(H)
actual_ground_state_energy = eigenvalues[0]
print(actual_ground_state_energy)

-1.137310199914228


In [77]:
print(abs(actual_ground_state_energy-ground_state_energy))

0.0022139439142283557
